# BrownDye2 association-rate preparation

End-to-end preparation pipeline for a BrownDye2 simulation of a docked
complex (`complex.pdb`) approaching a protein substrate (`substrate.pdb`).
Everything except the long-running BrownDye trajectory step lives in this
notebook.

Steps:

1. Verify protonation and fix the complex protein chain (PropKa + PDBFixer).
2. Assign ligand bond orders from SMILES and write an SDF for antechamber.
3. Parameterise the complex with AmberTools/GAFF2 (`pdb4amber`, `obabel`,
   `antechamber`, `parmchk2`, `tleap`, ParmEd PQR export).
4. Solve APBS on the complex PQR.
5. Parameterise the protein-only substrate with PDB2PQR + PropKa.
6. Solve APBS on the substrate PQR.
7. Build BrownDye XML inputs (`pqr2xml`, contact types, `make_rxn_pairs`,
   `make_rxn_file`, `bd_top`).
8. Run BrownDye trajectories: `bash bdrun.sh` from a terminal (kept as a
   shell script because it can take hours).

Activate the AmberTools environment before launching Jupyter:

```bash
conda activate ambertools
```

In [ ]:
import os
import shutil
from pathlib import Path

from Bio.PDB import PDBIO, PDBParser
from rdkit import Chem

from mdpp.prep import (
    BrownDyeBody,
    BrownDyeSolvent,
    ChainSelect,
    assign_topology,
    fix_pdb,
    infer_debye_length,
    run_propka,
    write_apbs_input,
    write_contact_types,
    write_input_xml,
)

# Inputs (this directory).
EXAMPLE_DIR = Path.cwd()
COMPLEX_PDB = EXAMPLE_DIR / "complex.pdb"
SUBSTRATE_PDB = EXAMPLE_DIR / "substrate.pdb"

# Workspace layout: each stage owns a folder under tmp/, with transient
# tool files kept in <stage>/intermediate/.
TMP_ROOT = EXAMPLE_DIR / "tmp"
STEP_DIR = TMP_ROOT / "complex" / "prep"
INTERMEDIATE_DIR = STEP_DIR / "intermediate"
COMPLEX_AMBERTOOLS_DIR = TMP_ROOT / "complex" / "ambertools"
COMPLEX_AMBERTOOLS_INTERMEDIATE = COMPLEX_AMBERTOOLS_DIR / "intermediate"
COMPLEX_APBS_DIR = TMP_ROOT / "complex" / "apbs"
COMPLEX_APBS_INTERMEDIATE = COMPLEX_APBS_DIR / "intermediate"
SUBSTRATE_PDB2PQR_DIR = TMP_ROOT / "substrate" / "pdb2pqr"
SUBSTRATE_PDB2PQR_INTERMEDIATE = SUBSTRATE_PDB2PQR_DIR / "intermediate"
SUBSTRATE_APBS_DIR = TMP_ROOT / "substrate" / "apbs"
SUBSTRATE_APBS_INTERMEDIATE = SUBSTRATE_APBS_DIR / "intermediate"
BDPREP_DIR = TMP_ROOT / "bdprep"
BDPREP_INTERMEDIATE = BDPREP_DIR / "intermediate"
for path in (
    STEP_DIR,
    INTERMEDIATE_DIR,
    COMPLEX_AMBERTOOLS_DIR,
    COMPLEX_AMBERTOOLS_INTERMEDIATE,
    COMPLEX_APBS_DIR,
    COMPLEX_APBS_INTERMEDIATE,
    SUBSTRATE_PDB2PQR_DIR,
    SUBSTRATE_PDB2PQR_INTERMEDIATE,
    SUBSTRATE_APBS_DIR,
    SUBSTRATE_APBS_INTERMEDIATE,
    BDPREP_DIR,
    BDPREP_INTERMEDIATE,
):
    path.mkdir(parents=True, exist_ok=True)

# Ligand template (used to assign bond orders to the PDB ligand block).
LIGAND_SMILES = (
    r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)"
    r"[C@@H](O)[C@H]3O"
)
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

# AmberTools (complex parameterisation).
PROTEIN_FF = "leaprc.protein.ff19SB"
LIGAND_FF = "leaprc.gaff2"
PB_RADII = "mbondi3"
STRIP_PROTEIN_H = True

# PDB2PQR (substrate parameterisation).
PDB2PQR_FF = "AMBER"

# APBS / BrownDye physics defaults live in mdpp.prep.apbs / mdpp.prep.browndye;
# override per-call below if a system needs non-standard values.

# BrownDye install + bodies.
BD_HOME = Path(os.environ.get("BD_HOME", "/apps/browndye2"))
BD_BIN = Path(os.environ.get("BD_BIN", str(BD_HOME / "bin")))
BD_AUX = Path(os.environ.get("BD_AUX", str(BD_HOME / "aux")))
CORE0 = "complex"
CORE1 = "substrate"
CORE0_IS_PROTEIN = True
CORE1_IS_PROTEIN = True
CORE0_DIELECTRIC = 4.0
CORE1_DIELECTRIC = 4.0

# BrownDye reaction criteria + run parameters consumed by bdrun.sh.
RXN_SEARCH_DISTANCE = 5.5
RXN_DISTANCE = 10.0
RXN_NEEDED = 3
N_THREADS = 1
SEED = 11111111
N_TRAJECTORIES = 100_000
N_TRAJECTORIES_PER_OUTPUT = 10
MAX_N_STEPS = 1_000_000
RESULTS_FILE = "results.xml"
TRAJECTORY_FILE = "trajectory"  # base name for trajectory*.xml / .index.xml dumps
N_STEPS_PER_OUTPUT = 10  # BD-step stride when recording trajectories (1 = every step)
DEBYE_LENGTH: float | None = None  # inferred from APBS logs in Step 7

# Export every shell-facing knob so subsequent %%bash cells inherit the
# same configuration without having to re-type defaults.
os.environ.update({
    "TMP_ROOT": str(TMP_ROOT),
    "COMPLEX_AMBERTOOLS_DIR": str(COMPLEX_AMBERTOOLS_DIR),
    "COMPLEX_AMBERTOOLS_INTERMEDIATE": str(COMPLEX_AMBERTOOLS_INTERMEDIATE),
    "COMPLEX_APBS_DIR": str(COMPLEX_APBS_DIR),
    "COMPLEX_APBS_INTERMEDIATE": str(COMPLEX_APBS_INTERMEDIATE),
    "SUBSTRATE_PDB2PQR_DIR": str(SUBSTRATE_PDB2PQR_DIR),
    "SUBSTRATE_PDB2PQR_INTERMEDIATE": str(SUBSTRATE_PDB2PQR_INTERMEDIATE),
    "SUBSTRATE_APBS_DIR": str(SUBSTRATE_APBS_DIR),
    "SUBSTRATE_APBS_INTERMEDIATE": str(SUBSTRATE_APBS_INTERMEDIATE),
    "BDPREP_DIR": str(BDPREP_DIR),
    "BDPREP_INTERMEDIATE": str(BDPREP_INTERMEDIATE),
    "SUBSTRATE_PDB": str(SUBSTRATE_PDB),
    "PROTEIN_FF": PROTEIN_FF,
    "LIGAND_FF": LIGAND_FF,
    "PB_RADII": PB_RADII,
    "STRIP_PROTEIN_H": "1" if STRIP_PROTEIN_H else "0",
    "PDB2PQR_FF": PDB2PQR_FF,
    "PH": f"{PH}",
    "BD_HOME": str(BD_HOME),
    "BD_BIN": str(BD_BIN),
    "BD_AUX": str(BD_AUX),
    "CORE0": CORE0,
    "CORE1": CORE1,
    "RXN_SEARCH_DISTANCE": f"{RXN_SEARCH_DISTANCE}",
    "RXN_DISTANCE": f"{RXN_DISTANCE}",
    "RXN_NEEDED": f"{RXN_NEEDED}",
})
os.environ["PATH"] = f"{BD_BIN}:{BD_AUX}:{os.environ.get('PATH', '')}"

## Step 1: Verify protonation and fix protein

Extract the protein chain, run PropKa as an explicit pKa/protonation-state check at the target pH, then add missing residues/atoms/hydrogens via PDBFixer.

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]
chains = {chain.id: chain for chain in model}

pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract protein chain into this step's intermediate folder.
protein_pdb = INTERMEDIATE_DIR / "protein.pdb"
pdb_io.save(str(protein_pdb), ChainSelect(PROTEIN_CHAIN_ID))

# Verify protonation with PropKa before PDBFixer assigns hydrogens.
propka_result = run_propka(protein_pdb)
nonstandard = propka_result.get_nonstandard(PH)
propka_report = STEP_DIR / "protein_propka.tsv"
with propka_report.open("w") as f:
    f.write(
        "residue_type\tres_num\tchain_id\tpka\tmodel_pka\tpropka_protonated\tmodel_protonated\n"
    )
    for residue in propka_result.residues:
        f.write(
            f"{residue.residue_type}\t{residue.res_num}\t{residue.chain_id}\t"
            f"{residue.pka:.3f}\t{residue.model_pka:.3f}\t"
            f"{residue.is_protonated_at(PH)}\t{residue.is_default_protonated_at(PH)}\n"
        )

print(f"PropKa residues checked: {len(propka_result.residues)} -> {propka_report}")
if nonstandard:
    print(f"PropKa predicts {len(nonstandard)} non-standard protonation state(s) at pH {PH}:")
    for residue in nonstandard:
        print(
            f"  {residue.label}: pKa={residue.pka:.2f}, model={residue.model_pka:.2f}, "
            f"PropKa={'protonated' if residue.is_protonated_at(PH) else 'deprotonated'}, "
            f"model={'protonated' if residue.is_default_protonated_at(PH) else 'deprotonated'}"
        )
else:
    print(f"PropKa agrees with model-pKa defaults for all titratable residues at pH {PH}.")

protein_fixed_pdb = STEP_DIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 2: Assign ligand topology

Extract ligand chain, assign bond orders from a SMILES template, and write an SDF
for antechamber. The SMILES is needed because PDB files lack bond-order information.

In [ ]:
# Extract ligand chain to the step intermediate folder for RDKit parsing.
ligand_pdb = INTERMEDIATE_DIR / "ligand.pdb"
pdb_io.save(str(ligand_pdb), ChainSelect(LIGAND_CHAIN_ID))

# Assign bond orders from SMILES template.
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"Ligand net charge: {ligand_net_charge}")

mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
mol_assigned = assign_topology(mol, template_mol)

# Write SDF for antechamber and metadata for the later AmberTools cells.
lig_resnames = sorted({res.get_resname().strip() for res in chains[LIGAND_CHAIN_ID].get_residues()})
lig_resname = lig_resnames[0]
mol_assigned.SetProp("_Name", lig_resname)

ligand_sdf = STEP_DIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)

(STEP_DIR / "ligand_charge.txt").write_text(f"{ligand_net_charge}\n")
(STEP_DIR / "ligand_resname.txt").write_text(f"{lig_resname}\n")

# Make resname + net charge available to subsequent bash cells.
os.environ["LIG_RESNAME"] = lig_resname
os.environ["NET_CHARGE"] = str(ligand_net_charge)
print(f"Ligand SDF -> {ligand_sdf}")
print(f"Ligand metadata -> {STEP_DIR / 'ligand_charge.txt'}, {STEP_DIR / 'ligand_resname.txt'}")

## Step 3: Parameterise the complex with AmberTools

Run the standard AmberTools pipeline on the prepared protein and ligand:

1. `pdb4amber` cleans the protein PDB for tleap.
2. `obabel` converts the ligand SDF to mol2 as an antechamber seed; we then
   patch the residue name to match the ligand chain.
3. `antechamber` assigns AM1-BCC charges and GAFF2 atom types.
4. `parmchk2` generates any missing GAFF2 parameters.
5. `tleap` combines the protein (ff19SB) and ligand (GAFF2) into a complex
   topology and saves prmtop/rst7 for protein, ligand, and complex bodies.
6. ParmEd exports each topology to PQR (charge + radius per atom).

Outputs live in `$COMPLEX_AMBERTOOLS_DIR`; transient files in its
`intermediate/` folder.

In [ ]:
# Stage the Step 1/2 outputs into the AmberTools intermediate folder.
shutil.copy(STEP_DIR / "protein_fixed.pdb", COMPLEX_AMBERTOOLS_INTERMEDIATE / "protein_fixed.pdb")
shutil.copy(STEP_DIR / "ligand.sdf", COMPLEX_AMBERTOOLS_INTERMEDIATE / "ligand.sdf")

In [ ]:
%%bash
set -euo pipefail
cd "$COMPLEX_AMBERTOOLS_INTERMEDIATE"

echo "=== 1. pdb4amber ==="
pdb4amber_args=(-i protein_fixed.pdb -o protein_amber.pdb -d --no-conect)
if [[ "$STRIP_PROTEIN_H" == "1" ]]; then
    pdb4amber_args+=(-y)
fi
pdb4amber "${pdb4amber_args[@]}"

echo "=== 2a. obabel: SDF -> mol2 ==="
obabel ligand.sdf -O ligand_seed.mol2

In [ ]:
# Patch the residue name written by obabel so it matches the docked ligand.
ligand_seed_mol2 = COMPLEX_AMBERTOOLS_INTERMEDIATE / "ligand_seed.mol2"
text = ligand_seed_mol2.read_text()
for old in ("UNL1", "UNL", "UNK"):
    text = text.replace(old, lig_resname)
if lig_resname not in text:
    raise RuntimeError(
        f"Residue name {lig_resname!r} not found in {ligand_seed_mol2} after replacement"
    )
ligand_seed_mol2.write_text(text)
print(f"Patched residue name -> {lig_resname}")

In [ ]:
%%bash
set -euo pipefail
cd "$COMPLEX_AMBERTOOLS_INTERMEDIATE"

echo "=== 2b. antechamber (AM1-BCC + GAFF2) ==="
antechamber \
    -i ligand_seed.mol2 -fi mol2 \
    -o ligand_amber.mol2 -fo mol2 \
    -c bcc -s 2 -at gaff2 \
    -nc "$NET_CHARGE" -rn "$LIG_RESNAME"

echo "=== 2c. parmchk2 (missing GAFF2 parameters) ==="
parmchk2 -i ligand_amber.mol2 -f mol2 -o ligand.frcmod

echo "=== 3. tleap (ff19SB protein + GAFF2 ligand -> complex) ==="
cat >tleap.in <<EOF
source $PROTEIN_FF
source $LIGAND_FF

$LIG_RESNAME = loadmol2 ligand_amber.mol2
loadamberparams ligand.frcmod
protein = loadpdb protein_amber.pdb
complex = combine {protein $LIG_RESNAME}

set default PBRadii $PB_RADII
saveamberparm protein protein.prmtop protein.rst7
saveamberparm $LIG_RESNAME ligand.prmtop ligand.rst7
saveamberparm complex complex.prmtop complex.rst7
quit
EOF
tleap -f tleap.in

In [ ]:
# ParmEd: convert each topology/coord pair to PQR (charge + radius per atom).
import parmed as pmd

for stem in ("protein", "ligand", "complex"):
    parm = pmd.load_file(
        str(COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.prmtop"),
        xyz=str(COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.rst7"),
    )
    parm.save(str(COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.pqr"), overwrite=True)
    for suffix in ("prmtop", "rst7", "pqr"):
        shutil.copy(
            COMPLEX_AMBERTOOLS_INTERMEDIATE / f"{stem}.{suffix}",
            COMPLEX_AMBERTOOLS_DIR / f"{stem}.{suffix}",
        )

print("Complex AmberTools outputs:")
for stem in ("protein", "ligand", "complex"):
    print("  ", COMPLEX_AMBERTOOLS_DIR / f"{stem}.pqr")

## Step 4: Solve APBS for the complex

Generate an APBS input file (`complex.in`) for the complex PQR and run
`apbs` to produce the electrostatic potential map `complex.dx`. The grid
is sized from the radius-inflated atom bounding box of `complex.pqr` and
padded by `FINE_PADDING` / `COARSE_PADDING` (set in the globals cell).

Protein-only and ligand-only diagnostic maps are optional - rerun
`write_apbs_input("protein", ...)` / `write_apbs_input("ligand", ...)` and
the bash cell with a different `STEMS=` if you want them.

In [ ]:
# Stage complex.pqr into the APBS intermediate folder and write complex.in.
shutil.copy(
    COMPLEX_AMBERTOOLS_DIR / "complex.pqr",
    COMPLEX_APBS_INTERMEDIATE / "complex.pqr",
)
apbs_in = write_apbs_input("complex", COMPLEX_APBS_INTERMEDIATE)
print(f"APBS input -> {apbs_in}")

# Optional diagnostics (uncomment to also map protein and ligand alone):
# for diag in ("protein", "ligand"):
#     shutil.copy(COMPLEX_AMBERTOOLS_DIR / f"{diag}.pqr", COMPLEX_APBS_INTERMEDIATE / f"{diag}.pqr")
#     write_apbs_input(diag, COMPLEX_APBS_INTERMEDIATE)

os.environ["STEMS"] = "complex"

In [ ]:
%%bash
set -euo pipefail
cd "$COMPLEX_APBS_INTERMEDIATE"

for stem in $STEMS; do
    echo "=== APBS: $stem ==="
    apbs "$stem.in" 2>&1 | tee "$stem.apbs.log"

    if [[ -s "$stem-PE0.dx" ]]; then
        mv "$stem-PE0.dx" "$stem.dx"
    elif [[ -s "$stem.pqr.dx" ]]; then
        mv "$stem.pqr.dx" "$stem.dx"
    fi

    if [[ ! -s "$stem.dx" ]]; then
        echo "Missing $stem.dx" >&2
        exit 1
    fi
    cp "$stem.in" "$COMPLEX_APBS_DIR/$stem.in"
    cp "$stem.apbs.log" "$COMPLEX_APBS_DIR/$stem.apbs.log"
    cp "$stem.dx" "$COMPLEX_APBS_DIR/$stem.dx"
done

ls -lh "$COMPLEX_APBS_DIR"/*.dx

## Step 5: Parameterise the substrate with PDB2PQR

PDB2PQR 3.7.1 does not expose an ff19SB option, so the substrate uses
PDB2PQR's bundled AMBER force field together with PropKa for pH-dependent
titration. The complex body still uses AmberTools/tleap ff19SB.

In [ ]:
%%bash
set -euo pipefail
cp "$SUBSTRATE_PDB" "$SUBSTRATE_PDB2PQR_INTERMEDIATE/substrate.pdb"
cd "$SUBSTRATE_PDB2PQR_INTERMEDIATE"

echo "=== PDB2PQR substrate ==="
pdb2pqr \
    --ff "$PDB2PQR_FF" \
    --ffout "$PDB2PQR_FF" \
    --keep-chain \
    --drop-water \
    --titration-state-method propka \
    --with-ph "$PH" \
    substrate.pdb \
    substrate.pqr 2>&1 | tee substrate.pdb2pqr.log

cp substrate.pqr "$SUBSTRATE_PDB2PQR_DIR/substrate.pqr"
cp substrate.pdb2pqr.log "$SUBSTRATE_PDB2PQR_DIR/substrate.pdb2pqr.log"
ls -lh "$SUBSTRATE_PDB2PQR_DIR/substrate.pqr"

## Step 6: Solve APBS for the substrate

Same flow as Step 4, applied to the substrate PQR.

In [ ]:
shutil.copy(
    SUBSTRATE_PDB2PQR_DIR / "substrate.pqr",
    SUBSTRATE_APBS_INTERMEDIATE / "substrate.pqr",
)
apbs_in = write_apbs_input("substrate", SUBSTRATE_APBS_INTERMEDIATE)
print(f"APBS input -> {apbs_in}")

In [ ]:
%%bash
set -euo pipefail
cd "$SUBSTRATE_APBS_INTERMEDIATE"

stem=substrate
echo "=== APBS: $stem ==="
apbs "$stem.in" 2>&1 | tee "$stem.apbs.log"

if [[ -s "$stem-PE0.dx" ]]; then
    mv "$stem-PE0.dx" "$stem.dx"
elif [[ -s "$stem.pqr.dx" ]]; then
    mv "$stem.pqr.dx" "$stem.dx"
fi

if [[ ! -s "$stem.dx" ]]; then
    echo "Missing $stem.dx" >&2
    exit 1
fi
cp "$stem.in" "$SUBSTRATE_APBS_DIR/$stem.in"
cp "$stem.apbs.log" "$SUBSTRATE_APBS_DIR/$stem.apbs.log"
cp "$stem.dx" "$SUBSTRATE_APBS_DIR/$stem.dx"
ls -lh "$SUBSTRATE_APBS_DIR/$stem.dx"

## Step 7: Build BrownDye XML inputs

Stage the complex and substrate PQR/DX files into `$BDPREP_INTERMEDIATE`,
infer the Debye length from the APBS logs, then walk the BrownDye setup
pipeline:

1. `pqr2xml` converts each PQR to BrownDye `atoms.xml`.
2. `contact_types.xml` lists every unique heavy-atom name per body.
3. `make_rxn_pairs` finds bound-pose contact pairs within `RXN_SEARCH_DISTANCE`.
4. `make_rxn_file` turns them into a reaction definition that fires when
   `RXN_NEEDED` pairs come within `RXN_DISTANCE` of each other.
5. `input.xml` is the BrownDye top-level config consumed by `bd_top`,
   which validates and writes `${CORE0}_${CORE1}_simulation.xml`.

`bdrun.sh` (kept as a shell script for the long-running trajectory step)
reads `$BDPREP_INTERMEDIATE/${CORE0}_${CORE1}_simulation.xml` directly.

In [ ]:
# Stage the four BrownDye inputs.
shutil.copy(COMPLEX_AMBERTOOLS_DIR / f"{CORE0}.pqr", BDPREP_INTERMEDIATE / f"{CORE0}.pqr")
shutil.copy(SUBSTRATE_PDB2PQR_DIR / f"{CORE1}.pqr", BDPREP_INTERMEDIATE / f"{CORE1}.pqr")
shutil.copy(COMPLEX_APBS_DIR / f"{CORE0}.dx", BDPREP_INTERMEDIATE / f"{CORE0}.dx")
shutil.copy(SUBSTRATE_APBS_DIR / f"{CORE1}.dx", BDPREP_INTERMEDIATE / f"{CORE1}.dx")

# Infer Debye length from the APBS logs unless the user pinned it manually.
if DEBYE_LENGTH is None:
    DEBYE_LENGTH = infer_debye_length(
        COMPLEX_APBS_DIR / f"{CORE0}.apbs.log",
        SUBSTRATE_APBS_DIR / f"{CORE1}.apbs.log",
    )
os.environ["DEBYE_LENGTH"] = f"{DEBYE_LENGTH}"
print(f"Debye length: {DEBYE_LENGTH} A")

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== 1. PQR -> BrownDye XML ==="
pqr2xml <"${CORE0}.pqr" >"${CORE0}_atoms.xml"
pqr2xml <"${CORE1}.pqr" >"${CORE1}_atoms.xml"

In [ ]:
# Heavy-atom contact criteria for make_rxn_pairs.
contact_types_path = write_contact_types(
    BDPREP_INTERMEDIATE / f"{CORE0}.pqr",
    BDPREP_INTERMEDIATE / f"{CORE1}.pqr",
    BDPREP_INTERMEDIATE / "contact_types.xml",
)
print("contact_types.xml ->", contact_types_path)

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== 2. Reaction criteria ==="
make_rxn_pairs -nonred \
    -mol0 "${CORE0}_atoms.xml" \
    -mol1 "${CORE1}_atoms.xml" \
    -ctypes contact_types.xml \
    -dist "$RXN_SEARCH_DISTANCE" >reaction_pairs.xml
make_rxn_file \
    -pairs reaction_pairs.xml \
    -state_from before \
    -state_to after \
    -rxn association \
    -mol0 "$CORE0" "$CORE0" \
    -mol1 "$CORE1" "$CORE1" \
    -distance "$RXN_DISTANCE" \
    -nneeded "$RXN_NEEDED" >reactions.xml

In [ ]:
# BrownDye top-level input consumed by bd_top.
input_xml = write_input_xml(
    BDPREP_INTERMEDIATE / "input.xml",
    BrownDyeBody(
        name=CORE0,
        atoms_xml=f"{CORE0}_atoms.xml",
        grid_dx=f"{CORE0}.dx",
        is_protein=CORE0_IS_PROTEIN,
        dielectric=CORE0_DIELECTRIC,
    ),
    BrownDyeBody(
        name=CORE1,
        atoms_xml=f"{CORE1}_atoms.xml",
        grid_dx=f"{CORE1}.dx",
        is_protein=CORE1_IS_PROTEIN,
        dielectric=CORE1_DIELECTRIC,
    ),
    solvent=BrownDyeSolvent(debye_length_a=DEBYE_LENGTH),
    n_threads=N_THREADS,
    seed=SEED,
    n_trajectories=N_TRAJECTORIES,
    n_trajectories_per_output=N_TRAJECTORIES_PER_OUTPUT,
    max_n_steps=MAX_N_STEPS,
    n_steps_per_output=N_STEPS_PER_OUTPUT,
    results_file=RESULTS_FILE,
    trajectory_file=TRAJECTORY_FILE,
)
print("input.xml ->", input_xml)

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== 3. BrownDye top-level input ==="
bd_top input.xml

for file in \
    "${CORE0}_atoms.xml" \
    "${CORE1}_atoms.xml" \
    contact_types.xml \
    reaction_pairs.xml \
    reactions.xml \
    input.xml \
    "${CORE0}_${CORE1}_simulation.xml"; do
    cp "$file" "$BDPREP_DIR/$file"
done

echo "Simulation XML: $BDPREP_DIR/${CORE0}_${CORE1}_simulation.xml"

## Step 8: Run BrownDye trajectories

Trajectory propagation is the only long-running step (`N_TRAJECTORIES=100_000`
with trajectory recording enabled can take many hours; weighted-ensemble mode
even longer), so it is **kept as a shell script**. From a terminal in this
directory:

```bash
conda activate ambertools
cd examples/browndye
bash bdrun.sh            # standard NAM mode
# or
MODE=we bash bdrun.sh    # weighted-ensemble mode
```

`bdrun.sh` reads `tmp/bdprep/intermediate/${CORE0}_${CORE1}_simulation.xml`
(populated by Step 7), runs `nam_simulation` or `we_simulation`, and writes
`tmp/bdrun/results.xml` plus `tmp/bdrun/rate_constant.txt`.

For APBS surface visualisation see `viz_complex_apbs.pml` / `viz_complex_apbs.cxc`
(PyMOL / ChimeraX). For a per-trajectory animation, continue with Step 9 below.

## Step 9: Visualise a reactive trajectory

When `<trajectory_file>` and `<n_steps_per_output>` are present in `input.xml`
(Step 7), `nam_simulation` writes `trajectoryN.xml` plus `trajectoryN.index.xml`
(one pair per thread) into `tmp/bdrun/intermediate/`. To turn one reactive
trajectory into a VTF animation for VMD:

1. `process_trajectories -srxn association` lists the indices of trajectories
   that reach the `association` state.
2. `process_trajectories -n <idx> -nstride <stride>` decompresses one
   trajectory into `trajectory.xml`; `-nstride` thins the configurations
   further to keep the file manageable.
3. `vtf_trajectory` converts `trajectory.xml` to `trajectory.vtf` for VMD
   (`vmd trajectory.vtf`). Use `-trial` first if you want a file-size estimate
   before generating the full VTF.

Tune via env vars before running the cell:

- `TRAJ_THREAD` (default `0`) - which `trajectory{N}.xml` to read.
- `TRAJ_IDX` (default: first listed reactive trajectory) - which trajectory inside the file.
- `STRIDE` (default `10`) - additional `-nstride` thinning applied by `process_trajectories`.


In [ ]:
%%bash
set -euo pipefail
BDRUN_DIR="${BDRUN_DIR:-$TMP_ROOT/bdrun}"
BDRUN_INTERMEDIATE="${BDRUN_INTERMEDIATE:-$BDRUN_DIR/intermediate}"
TRAJ_THREAD="${TRAJ_THREAD:-0}"
STRIDE="${STRIDE:-10}"

traj_xml="trajectory${TRAJ_THREAD}.xml"
traj_index="trajectory${TRAJ_THREAD}.index.xml"

cd "$BDRUN_INTERMEDIATE"
if [[ ! -s "$traj_xml" || ! -s "$traj_index" ]]; then
    echo "Missing $traj_xml or $traj_index in $BDRUN_INTERMEDIATE" >&2
    echo "Re-run bdrun.sh after Step 7 was executed with <trajectory_file>." >&2
    exit 1
fi

echo "=== Reactive trajectories in $traj_xml ==="
process_trajectories \
    -traj "$traj_xml" \
    -index "$traj_index" \
    -srxn association | tee reactive_trajectories.txt

# Default to the first reactive trajectory number unless TRAJ_IDX is set.
TRAJ_IDX="${TRAJ_IDX:-$(grep -oE '[0-9]+' reactive_trajectories.txt | head -n1 || true)}"
if [[ -z "$TRAJ_IDX" ]]; then
    echo "No reactive trajectories found in $traj_xml" >&2
    exit 1
fi
echo "Selected reactive trajectory index=$TRAJ_IDX, stride=$STRIDE"

echo "=== Decompress to trajectory.xml ==="
process_trajectories \
    -traj "$traj_xml" \
    -index "$traj_index" \
    -n "$TRAJ_IDX" \
    -nstride "$STRIDE" >trajectory.xml

echo "=== Estimate VTF size ==="
vtf_trajectory \
    -mol0 "${CORE0}_atoms.xml" \
    -mol1 "${CORE1}_atoms.xml" \
    -trajf trajectory.xml \
    -trial || true

echo "=== Generate trajectory.vtf for VMD ==="
vtf_trajectory -traj trajectory.xml >trajectory.vtf

mkdir -p "$BDRUN_DIR"
cp reactive_trajectories.txt "$BDRUN_DIR/reactive_trajectories.txt"
cp trajectory.xml "$BDRUN_DIR/trajectory.xml"
cp trajectory.vtf "$BDRUN_DIR/trajectory.vtf"
echo
ls -lh "$BDRUN_DIR/trajectory.vtf"
echo "Load in VMD with:  vmd $BDRUN_DIR/trajectory.vtf"
